# 07 — Polars : DataFrames modernes et performants

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- créer des DataFrames Polars
- utiliser les **expressions** Polars
- comprendre le mode **lazy** et les optimisations
- comparer avec Pandas

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- toute la formation Intermédiaire (OO, typage, tests, SQL)
- `Decimal`, `Fraction`, `statistics`, `random`/`secrets`
- Pandas

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- rien de spécifique

## Plan

1. Pourquoi Polars
2. Créer un DataFrame
3. Expressions
4. Filtrage et sélection
5. Groupby et agrégation
6. Mode lazy
7. Polars vs Pandas
8. Synthèse
9. Exercices

---

## 1. Pourquoi Polars

Polars est une bibliothèque de DataFrames écrite en **Rust**, 5-50x plus rapide que Pandas pour beaucoup d'opérations. Son API est basée sur des **expressions** composables, et le mode **lazy** permet des optimisations automatiques.

---

## 2. Créer un DataFrame

In [ ]:
import polars as pl

df = pl.DataFrame({
    'nom': ['Mars', 'Venus', 'Io'],
    'capacite': [12, 6, 15],
    'prix_h': [50.0, 30.0, 80.0],
})
df


---

## 3. Expressions

Les expressions Polars sont le cœur de l'API. Elles sont composables et lazy.

In [ ]:
df.select(pl.col('nom'), pl.col('prix_h') * 1.2)


In [ ]:
df.with_columns(prix_ttc=pl.col('prix_h') * 1.2)


In [ ]:
df.with_columns(pl.col('nom').str.to_uppercase().alias('NOM'))


---

## 4. Filtrage et sélection

In [ ]:
df.filter(pl.col('capacite') >= 10)


In [ ]:
df.filter((pl.col('capacite') >= 10) & (pl.col('prix_h') < 100))


---

## 5. Groupby et agrégation

In [ ]:
reservations = pl.DataFrame({
    'salle': ['Mars', 'Venus', 'Mars', 'Io', 'Mars'],
    'heures': [2, 1, 3, 2, 1],
})
reservations.group_by('salle').agg(pl.col('heures').sum())


---

## 6. Mode lazy

Le mode lazy construit un **plan d'exécution** optimisé avant de lancer le calcul.

In [ ]:
q = (
    df.lazy()
    .filter(pl.col('capacite') >= 10)
    .with_columns(prix_ttc=pl.col('prix_h') * 1.2)
    .select('nom', 'prix_ttc')
)
q.explain()  # plan d'exécution


In [ ]:
q.collect()  # exécution effective


---

## 7. Polars vs Pandas

| Critère | Pandas | Polars |
|---|---|---|
| Langage | C/Cython | Rust |
| API | Méthodes sur colonnes | Expressions |
| Lazy | Non | Oui |
| Multi-threading | Limité | Natif |
| Maturité | ~20 ans | ~4 ans |
| Écosystème ML | Dominant | En croissance |

**Recommandation** : Polars pour les nouveaux projets data ; Pandas quand l'écosystème l'impose (scikit-learn, etc.).

---

## Synthèse

| API Polars | Rôle |
|---|---|
| `pl.DataFrame({...})` | Créer |
| `pl.col('x')` | Expression de colonne |
| `.select()`, `.with_columns()` | Sélection / ajout |
| `.filter()` | Filtrage |
| `.group_by().agg()` | Agrégation |
| `.lazy()` / `.collect()` | Mode lazy |


### Règles à retenir

1. **Expressions Polars > méthodes Pandas** : plus composables, plus optimisables.
2. **Mode lazy par défaut** pour les gros datasets.
3. **Polars pour la vitesse**, Pandas pour l'écosystème ML.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Créer et transformer *(facile)*

Créer un DataFrame Polars avec 3 colonnes. Ajouter une colonne calculée. Filtrer.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la correction</summary>

```python
import polars as pl
df = pl.DataFrame({'nom': ['A','B','C'], 'prix': [10,20,5], 'qte': [3,1,7]})
df = df.with_columns(total=pl.col('prix') * pl.col('qte'))
print(df.filter(pl.col('total') > 15))
```

</details>

### Exercice 2 — Lazy pipeline *(moyen)*

Écrire un pipeline lazy : lire un DataFrame, filtrer, grouper, collecter.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la correction</summary>

```python
import polars as pl
df = pl.DataFrame({'cat': ['A','B','A','B','A'], 'val': [10,20,30,40,50]})
result = (
    df.lazy()
    .filter(pl.col('val') > 15)
    .group_by('cat').agg(pl.col('val').sum())
    .collect()
)
print(result)
```

</details>

---

## Ressources externes

### Documentation officielle
- [Polars user guide](https://docs.pola.rs/)